# Notebook 14 – Performance Optimization
## Learning objectives
Make Pandas code faster and more memory-efficient using vectorization, appropriate data types, and efficient workflows.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "product": ["Pen", "Book", "Bag", "Pen", "Book"],
    "units": [10, 4, 2, 7, 6],
    "unit_price": [5.0, 80.0, 500.0, 5.0, 80.0]
})
print(df)

## 1. Vectorization
**Definition:** Vectorization performs an operation on an entire array or Series without writing an explicit Python loop for each value.

**Real-world:** Calculate totals for many receipts at once.
**Business:** Calculate revenue for thousands of sales rows.
**AI/ML:** Prepare numeric features efficiently.

The vectorized expression is concise and typically faster than a Python-level loop.

In [ ]:
df["revenue"] = df["units"] * df["unit_price"]
print(df)

## 2. `apply()` vs Loops
**Definition:** A loop processes items one at a time. `apply()` applies a function along a Series or DataFrame axis. Neither is automatically the fastest choice; built-in vectorized operations are often preferable.

**Real-world:** Label orders based on their amount.
**Business:** Apply a discount rule to order totals.
**AI/ML:** Transform features before model training.

In [ ]:
# Preferred for this simple numeric rule: vectorized operation
df["large_order"] = df["revenue"] >= 500

# apply() example for a rule that is easier to express as a function
df["revenue_label"] = df["revenue"].apply(lambda value: "High" if value >= 500 else "Regular")
print(df)

# A loop can work too, but is often more verbose for column-wise calculations
labels = []
for value in df["revenue"]:
    labels.append("High" if value >= 500 else "Regular")
print("Loop labels:", labels)

## 3. Memory Optimization
**Definition:** Memory optimization reduces the memory used to store data, often by selecting suitable data types and avoiding unnecessary copies.

**Real-world:** Fit a large transaction dataset into available RAM.
**Business:** Reduce the memory footprint of daily reports.
**AI/ML:** Load more training data within a machine's memory limits.

Inspect memory with `memory_usage(deep=True)`. Downcasting is useful only when the chosen type can represent the values safely.

In [ ]:
memory_before = df.memory_usage(deep=True).sum()
optimized = df.copy()
optimized["units"] = pd.to_numeric(optimized["units"], downcast="integer")
optimized["unit_price"] = pd.to_numeric(optimized["unit_price"], downcast="float")
memory_after = optimized.memory_usage(deep=True).sum()
print("Before:", memory_before, "bytes")
print("After: ", memory_after, "bytes")
print(optimized.dtypes)

## 4. Category Data Type
**Definition:** The `category` dtype stores repeated labels using a set of categories and coded values. It can reduce memory when a column has relatively few unique values.

**Real-world:** Store repeated product types.
**Business:** Store department or region labels.
**AI/ML:** Manage categorical columns before encoding them for a model.

It may not save memory when nearly every value is unique.

In [ ]:
category_df = df.copy()
print("Object memory before:", category_df["product"].memory_usage(deep=True))
category_df["product"] = category_df["product"].astype("category")
print("Category memory after:", category_df["product"].memory_usage(deep=True))
print(category_df["product"].dtype)

## 5. Efficient Pandas Practices
**Definition:** Efficient practices are coding choices that reduce unnecessary work and resource use.

- Select only needed columns when reading large files.
- Filter early when possible.
- Prefer vectorized built-in operations.
- Use `groupby()` and aggregation instead of manually looping over groups.
- Choose suitable numeric and categorical dtypes.
- Avoid repeatedly concatenating inside a loop; collect pieces and concatenate once.
- Use `read_csv(..., usecols=...)` and `chunksize=...` for large files.
- Avoid chained assignment; use `.loc` for explicit updates.
- Benchmark with representative data rather than assuming an optimization helps.

**Business example:** Aggregate total revenue by product.
**AI/ML example:** Load only the columns required for training.

In [ ]:
# Efficient group aggregation
summary = df.groupby("product", as_index=False).agg(
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum")
)
print(summary)

# Read only selected columns from a CSV:
# sales = pd.read_csv("sales.csv", usecols=["product", "units", "unit_price"])

# Process a large CSV in chunks:
# for chunk in pd.read_csv("large_sales.csv", chunksize=100_000):
#     print(chunk["units"].sum())